# DLA and topology-constrained differentiable control

This notebook exercises the public `scpn_quantum_control.dla_topology_control` facade. It projects a dense state into an exact computational-basis parity sector, optimises a synthetic parity-protected objective, differentiates a topology-ledger projection on one fixed smooth active set, and demonstrates a fail-closed non-smooth branch.

The results are finite local NumPy evidence. They are not a full DLA classification, controllability or error-correction certificate, persistent-homology derivative, provider or QPU result, or deployment instruction.

In [ ]:
import numpy as np

from scpn_quantum_control.dla_topology_control import (
    ParityProtectedQuadraticObjective,
    ParitySector,
    ParitySectorProjector,
    ProjectedGradientConfig,
    build_dla_topology_control_evidence,
    optimise_parity_protected_state,
    topology_projection_jvp,
    topology_projection_support,
    topology_projection_vjp,
)
from scpn_quantum_control.topology_control.constraints import (
    CouplingGraphBounds,
    TopologyConstraintLedger,
)

## 1. Optimise inside an exact parity sector

The forward projection delegates to the repository's existing DLA-parity projector. The JVP and VJP use the same self-adjoint linear projection, while the objective adds an analytic outside-sector leakage penalty. Projection happens inside every proposed gradient step.

In [ ]:
projector = ParitySectorProjector(3, ParitySector.EVEN)
target = np.zeros(projector.dimension, dtype=np.complex128)
target[0] = 1.0
objective = ParityProtectedQuadraticObjective(projector, target, leakage_weight=2.0)
initial = np.array([1.0, 0.8j, -0.4, 0.3, 0.2j, -0.6, 0.5, 0.1j])
trace = optimise_parity_protected_state(
    initial,
    objective,
    ProjectedGradientConfig(max_steps=40, initial_step_size=0.5),
)

print("initial objective", objective(initial))
print("final objective", objective(trace.final_state))
print("final leakage", projector.leakage_value_and_gradient(trace.final_state).value)
print("accepted steps", trace.accepted_steps)

## 2. Differentiate a fixed topology-ledger branch

This ledger uses a signed policy, inactive bounds, a fixed hardware mask, and one frozen edge. Those operations have a well-defined local affine derivative while their discrete identities remain fixed. The JVP and VJP satisfy the Frobenius adjoint identity.

In [ ]:
ledger = TopologyConstraintLedger(
    bounds=CouplingGraphBounds(-2.0, 2.0),
    sign_policy="signed",
    hardware_edges={(0, 1), (1, 2), (2, 3), (0, 3)},
    frozen_edges={(0, 1): 0.25},
)
matrix = np.array(
    [
        [0.0, 0.4, -0.6, 0.7],
        [0.2, 0.0, 0.5, -0.4],
        [-0.3, 0.8, 0.0, 0.6],
        [0.9, -0.7, 0.2, 0.0],
    ]
)
rng = np.random.default_rng(54)
tangent = rng.normal(size=(4, 4))
cotangent = rng.normal(size=(4, 4))
differential = topology_projection_jvp(ledger, matrix, tangent)
vjp = topology_projection_vjp(ledger, matrix, cotangent)
left = float(np.vdot(differential.projected_tangent, cotangent).real)
right = float(np.vdot(tangent, vjp).real)

print("derivative supported", differential.support.derivative_supported)
print("adjoint error", abs(left - right))
print("differential digest", differential.content_digest)

## 3. Inspect the fail-closed boundary

A zero-valued nonnegative branch lies at a sign kink and at the default lower clipping bound. The support report names both blockers instead of inventing a derivative. Positive algebraic-connectivity thresholds and active total-weight rescaling are likewise unsupported.

In [ ]:
non_smooth_ledger = TopologyConstraintLedger()
report = topology_projection_support(non_smooth_ledger, np.zeros((3, 3)))
print("derivative supported", report.derivative_supported)
print("blocking capabilities", report.blocking_capabilities)
print("support digest", report.content_digest)

## 4. Rebuild the deterministic evidence object

The evidence builder checks finite differences, the JVP/VJP adjoint identity, the projected synthetic optimisation, and composition with the existing hard topology optimiser. It performs no provider, QPU, hardware, or deployment action.

In [ ]:
evidence = build_dla_topology_control_evidence()
print("evidence digest", evidence.content_digest)
print("parity JVP max error", evidence.parity_jvp_max_abs_error)
print("topology JVP max error", evidence.topology_jvp_max_abs_error)
print("unsupported branches", evidence.unsupported_blockers)

## Next steps

Read the [DLA and topology-constrained differentiable control guide](../docs/dla_topology_constrained_control.md) for equations, shapes, errors, API details, evidence custody, citations, and non-claims. Byte-check the committed evidence with `PYTHONPATH=src:oscillatools/src python scripts/run_dla_topology_control_evidence.py --check`.